# ROAD ADV ATTACKS
**Function to load subset and filter normal samples**\
**Adversarial Samples Generation**\
**Constraint Compliance**

In [64]:
def load_dataset(files):
    fp_flag = 0
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    df=df[df['Flag'] == fp_flag].copy()
    X = df.drop(columns=['Flag'])   # features only
    y = df['Flag'].values           
    X.columns = [c.replace("[", "_").replace("]", "").replace("<", "_") for c in X.columns]
    return X, y


def generate_constrained_attack(estimator, X, method="FGSM", eps=1.0, eps_step=0.1, max_iter=10):
    """
    Generate adversarial samples with IVN constraints using FGSM, BIM, or PGD.
    
    Parameters:
    - estimator: ART classifier
    - X: np.ndarray or pd.DataFrame, shape (n_samples, 10)
    - method: str, one of {"FGSM", "BIM", "PGD"}
    - eps: float, total allowed perturbation
    - eps_step: float, step size
    - max_iter: int, only used for iterative attacks (BIM, PGD)
    
    Returns:
    - X_adv: adversarial samples (np.ndarray), clipped to [0, 255] on DATA[0]-[7]
    """
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()

    # Only allow perturbation on DATA[0]-[7]
    perturbation_mask = np.array([False, False] + [True] * 8)

    # Select the attack method
    if method == "FGSM":
        attack = FastGradientMethod(estimator=estimator, eps=eps, eps_step=eps_step)
    elif method == "BIM":
        attack = BasicIterativeMethod(estimator=estimator, eps=eps, eps_step=eps_step, max_iter=max_iter, verbose=False)
    elif method == "PGD":
        attack = ProjectedGradientDescent(estimator=estimator, eps=eps, eps_step=eps_step, max_iter=max_iter, num_random_init=1, verbose=False)
    else:
        raise ValueError("Unsupported attack method. Choose 'FGSM', 'BIM', or 'PGD'.")

    # Generate adversarial examples
    X_adv = attack.generate(x=X, mask=perturbation_mask)

    # Clip DATA[0] to DATA[7] to valid byte range
    X_adv[:, 2:] = np.clip(X_adv[:, 2:], 0, 255)
    
    # Log attack details
    print(f"[{method}] Generated adversarial examples with eps={eps}, eps_step={eps_step}, max_iter={max_iter if method != 'FGSM' else 'N/A'}")
    
    return X_adv

def constraint_compliant(X):
    X_np = X.to_numpy() if isinstance(X, pd.DataFrame) else X
    return (
        (X_np[:, 0] >= 0) & (X_np[:, 0] <= 1068) &  # CAN ID
        (X_np[:, 1] >= 1) & (X_np[:, 1] <= 8) &     # DLC
        np.all((X_np[:, 2:] >= 0) & (X_np[:, 2:] <= 255), axis=1)  # DATA[0]-[7]
    )


### Evaluate Adversarial Models
### Run all the attacks

In [ ]:
def evaluate_model_on_adversarial_tabular(
    model,
    X_clean,
    y_clean,          # <-- now you pass true labels (0=normal, 1=attack)
    X_adv_dict,
    model_name="RF",
    attack_name="BIM",
    is_probabilistic=False
):
    results = []
    sample_size = len(X_clean)

    # Predict on clean data
    # pred = model.predict(X_clean)
    # if is_probabilistic:
    #     y_pred_clean =  (pred > 0.5).astype(int) #np.argmax(model.predict(X_clean), axis=1)
    # else:
    #     y_pred_clean = pred

    if is_probabilistic:
        y_prob = model.predict(X_clean).ravel()   # values in (0, 1)
        y_pred_clean = (y_prob >= 0.5).astype(int) #y_pred_clean = np.argmax(model.predict(X_clean), axis=1)
    else:
        y_pred_clean = model.predict(X_clean)

    # Confusion matrix (TN, FP, FN, TP)
    cm_clean = confusion_matrix(y_clean, y_pred_clean, labels=[0,1])
    tn, fp, fn, tp = cm_clean.ravel()
    f1_clean = f1_score(y_clean, y_pred_clean, average='weighted')

    for eps_val, X_adv in X_adv_dict.items():
        valid_mask = constraint_compliant(X_adv)
        X_adv_valid = X_adv[valid_mask]
        if len(X_adv_valid) == 0:
            continue

        # Ground truth is same as y_clean for aligned samples
        y_true_adv = y_clean[valid_mask]

        # if is_probabilistic:
        #     y_pred_adv = np.argmax(model.predict(X_adv_valid), axis=1)
        # else:
        #     y_pred_adv = model.predict(X_adv_valid)

        if is_probabilistic:
            y_prob_adv = model.predict(X_adv_valid).ravel()   # values in (0, 1)
            y_pred_adv = (y_prob_adv >= 0.5).astype(int)
            #y_pred_adv = np.argmax(model.predict(X_adv_valid), axis=1) # For saving as softmax
            y_pred_adv_binary = (y_pred_adv.ravel() >= 0.5).astype(int)
        else:
            y_pred_adv = model.predict(X_adv_valid)
            y_pred_adv_binary = y_pred_adv.astype(int)




        y_pred_adv_binary = (y_pred_adv != 0).astype(int)
        f1_adv = f1_score(y_true_adv, y_pred_adv_binary, average='weighted')
        fp_adv = np.sum(y_pred_adv_binary) # Since the data is benign only otherwise fp_adv = np.sum((y_true_adv == 0) & (y_pred_adv_binary == 1))
        asr = fp_adv / len(X_adv_valid)


        results.append([
            model_name, sample_size,
            f"{f1_clean*100:.1f}%", fp,
            attack_name, eps_val,
            f"{f1_adv*100:.1f}%", fp_adv, f"{asr:.1%}"
        ])
    columns = ["Model", "Sample Size",
               "F1 Score", "FP", 
               "Attack", "Epsilon",
               "F1 Score (Adv)", "FP (Adv)", "ASR"]
    
    return pd.DataFrame(results, columns=columns)



def run_attacks(dataset_name, X_normal, y_clean, models, attack_methods):
    results = {}
    cols = X_normal.columns
    for attack in attack_methods:
        X_adv_dict_all = {}  # holds adversarial sets per model

        # === Generate adversarial examples ===
        if attack in ["FGSM", "BIM", "PGD"]:
            # These only use the DNN
            if attack == "FGSM":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="FGSM", eps=1.0, eps_step=0.1),
                    5: generate_constrained_attack(dnn_art, X_normal, method="FGSM", eps=5.0, eps_step=0.1)
                }
            elif attack == "BIM":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="BIM", eps=1.0, eps_step=0.1, max_iter=10),
                    5: generate_constrained_attack(dnn_art, X_normal, method="BIM", eps=5.0, eps_step=0.1, max_iter=10)
                }
            elif attack == "PGD":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="PGD", eps=1.0, eps_step=0.1, max_iter=20),
                    5: generate_constrained_attack(dnn_art, X_normal, method="PGD", eps=5.0, eps_step=0.1, max_iter=20)
                }

            # store once, reused by all models
            X_adv_dict_all = {"DNN": X_adv_dict}

        else:
            continue
    
        # === Evaluate across all models ===
        for model_name, model_obj in models.items():
            X_fixed = pd.DataFrame(X_normal.values, columns=cols)

            # choose the right adversarial examples
            if attack == "DT":
                X_adv_fixed = {eps: pd.DataFrame(X_adv, columns=cols)
                               for eps, X_adv in X_adv_dict_all[model_name].items()}
            else:
                X_adv_fixed = {eps: pd.DataFrame(X_adv, columns=cols)
                               for eps, X_adv in X_adv_dict_all["DNN"].items()}

            result_key = f"{dataset_name}_{model_name}_{attack}"
            results[result_key] = evaluate_model_on_adversarial_tabular(model_obj, X_fixed, y_clean, X_adv_fixed, 
                                model_name=model_name, attack_name=attack, is_probabilistic=(model_name == "DNN"))

    return results


# False Positives (FPs)

In [66]:
import glob
import os
import pandas as pd
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from art.attacks.evasion import FastGradientMethod, BasicIterativeMethod, ProjectedGradientDescent
from tensorflow.keras.models import load_model
from art.estimators.classification import KerasClassifier, SklearnClassifier
from art.estimators.classification import XGBoostClassifier
from art.estimators.classification import TensorFlowV2Classifier
from tensorflow.keras.optimizers import legacy as legacy_optimizers
import tensorflow as tf
from sklearn.metrics import f1_score, confusion_matrix


csv_folder = 'preprocessed'  # Update this
csv_files = sorted(glob.glob(os.path.join(csv_folder, "*.csv")))
csa = csv_files[:3]
fa = csv_files[3:6]
mecta = csv_files[6:7]
msa = csv_files[7:10]
rloffa = csv_files[10:13]
rlona = csv_files[13:16]
print(csa, fa,mecta,msa,rloffa,rlona)

# Load datasets with labels This lable is for FALSE POSITIVES #fp_flag = 0
# fp_flag = 0
X_csa, y_csa = load_dataset(csa)
X_fa, y_fa = load_dataset(fa)
X_mecta, y_mecta = load_dataset(mecta)
X_msa, y_msa = load_dataset(msa)
X_rloffa, y_rloffa = load_dataset(rloffa)
X_rlona, y_rlona = load_dataset(rlona)

# Load models
dnn_model = load_model("models/dnn_model.h5")
dt_model = joblib.load("models/dt_model.pkl")
rf_model = joblib.load("models/rf_model.pkl")
et_model = joblib.load("models/et_model.pkl")
xgb_model = joblib.load("models/xgboost_model.pkl")

#Wrap Model
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
dnn_art = TensorFlowV2Classifier(
    model=dnn_model,
    loss_object=loss_object,
    optimizer=legacy_optimizers.Adam(),
    nb_classes=2,
    input_shape=(X_csa.shape[1],),
    clip_values=(0, 255),
)


models = {"DNN": dnn_art, "DT": dt_model, "RF": rf_model, "ET": et_model, "XGBoost": xgb_model}
datasets = {"rlona": (X_rlona, y_rlona), "rloffa": (X_rloffa, y_rloffa), "msa": (X_msa, y_msa), "mecta": (X_mecta, y_mecta), "fa": (X_fa, y_fa), "csa": (X_csa,y_csa)}
attack_methods = ["FGSM", "BIM", "PGD"]

# mapping dataset keys to names
dataset_labels = {
    "csa": "Correlated Signal Attack",
    "fa": "Fuzzing Attack",
    "mecta": "Max Engine Coolant Temp Attack",
    "msa": "max_speedometer_attack",  
    "rloffa": "Reverse Light Off Attack",
    "rlona": "Reverse Light On Attack"
}

# Run attacks for all datasets
all_results_fp = {}
for dataset_key, (X,y) in datasets.items():
    dataset_results = run_attacks(dataset_key, X, y, models, attack_methods)

    # merge all model/attack results into one big DataFrame for this dataset
    merged_df = pd.concat(dataset_results.values(), ignore_index=True)

    # store under dataset key
    all_results_fp[dataset_key] = merged_df


['preprocessed/csa1.csv', 'preprocessed/csa2.csv', 'preprocessed/csa3.csv'] ['preprocessed/fa1.csv', 'preprocessed/fa2.csv', 'preprocessed/fa3.csv'] ['preprocessed/mecta.csv'] ['preprocessed/msa1.csv', 'preprocessed/msa2.csv', 'preprocessed/msa3.csv'] ['preprocessed/rloffa1.csv', 'preprocessed/rloffa2.csv', 'preprocessed/rloffa3.csv'] ['preprocessed/rlona1.csv', 'preprocessed/rlona2.csv', 'preprocessed/rlona3.csv']
[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=N/A
[FGSM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=N/A
[BIM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=10
[BIM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=10
[PGD] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=20
[PGD] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=20
[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=N/A
[FGSM] Generated adversarial example

# Table III - Table VII

In [67]:
for dataset_key, df in all_results_fp.items():
    dataset_label = dataset_labels.get(dataset_key, dataset_key)
    print(f"\n=== {dataset_label} ===")
    display(df)   # Jupyter; if terminal: print(df.to_string())


=== Reverse Light On Attack ===


,Model,Sample Size,F1 Score,FP,Attack,Epsilon,F1 Score (Adv),FP (Adv),ASR
0,DNN,480021,99.9%,702,FGSM,1,99.9%,462,0.1%
1,DNN,480021,99.9%,702,FGSM,5,99.9%,462,0.1%
2,DT,480021,99.8%,1737,FGSM,1,99.7%,2025,0.6%
3,DT,480021,99.8%,1737,FGSM,5,99.7%,2025,0.6%
4,RF,480021,99.8%,1824,FGSM,1,99.7%,1896,0.6%
5,RF,480021,99.8%,1824,FGSM,5,99.7%,1896,0.6%
6,ET,480021,99.8%,1737,FGSM,1,99.7%,1653,0.5%
7,ET,480021,99.8%,1737,FGSM,5,99.7%,1809,0.5%
8,XGBoost,480021,99.8%,1899,FGSM,1,99.7%,2025,0.6%
9,XGBoost,480021,99.8%,1899,FGSM,5,99.7%,2025,0.6%



=== Reverse Light Off Attack ===


,Model,Sample Size,F1 Score,FP,Attack,Epsilon,F1 Score (Adv),FP (Adv),ASR
0,DNN,180357,100.0%,6,FGSM,1,100.0%,6,0.0%
1,DNN,180357,100.0%,6,FGSM,5,100.0%,0,0.0%
2,DT,180357,100.0%,18,FGSM,1,100.0%,6,0.0%
3,DT,180357,100.0%,18,FGSM,5,100.0%,6,0.0%
4,RF,180357,100.0%,18,FGSM,1,100.0%,6,0.0%
5,RF,180357,100.0%,18,FGSM,5,100.0%,6,0.0%
6,ET,180357,100.0%,18,FGSM,1,100.0%,6,0.0%
7,ET,180357,100.0%,18,FGSM,5,100.0%,6,0.0%
8,XGBoost,180357,100.0%,18,FGSM,1,100.0%,6,0.0%
9,XGBoost,180357,100.0%,18,FGSM,5,100.0%,6,0.0%



=== max_speedometer_attack ===


...

,Model,Sample Size,F1 Score,FP,Attack,Epsilon,F1 Score (Adv),FP (Adv),ASR
0,DNN,57932,100.0%,0,FGSM,1,100.0%,0,0.0%
1,DNN,57932,100.0%,0,FGSM,5,100.0%,0,0.0%
2,DT,57932,100.0%,17,FGSM,1,100.0%,14,0.0%
3,DT,57932,100.0%,17,FGSM,5,100.0%,14,0.0%
4,RF,57932,100.0%,16,FGSM,1,100.0%,13,0.0%
5,RF,57932,100.0%,16,FGSM,5,100.0%,13,0.0%
6,ET,57932,100.0%,16,FGSM,1,100.0%,13,0.0%
7,ET,57932,100.0%,16,FGSM,5,100.0%,13,0.0%
8,XGBoost,57932,100.0%,16,FGSM,1,100.0%,13,0.0%
9,XGBoost,57932,100.0%,16,FGSM,5,100.0%,13,0.0%



=== Fuzzing Attack ===


,Model,Sample Size,F1 Score,FP,Attack,Epsilon,F1 Score (Adv),FP (Adv),ASR
0,DNN,87907,100.0%,19,FGSM,1,100.0%,6,0.0%
1,DNN,87907,100.0%,19,FGSM,5,100.0%,6,0.0%
2,DT,87907,100.0%,22,FGSM,1,99.9%,67,0.1%
3,DT,87907,100.0%,22,FGSM,5,99.9%,67,0.1%
4,RF,87907,100.0%,20,FGSM,1,100.0%,8,0.0%
5,RF,87907,100.0%,20,FGSM,5,100.0%,8,0.0%
6,ET,87907,100.0%,18,FGSM,1,100.0%,8,0.0%
7,ET,87907,100.0%,18,FGSM,5,100.0%,8,0.0%
8,XGBoost,87907,100.0%,22,FGSM,1,99.9%,69,0.1%
9,XGBoost,87907,100.0%,22,FGSM,5,99.9%,69,0.1%



=== Correlated Signal Attack ===


,Model,Sample Size,F1 Score,FP,Attack,Epsilon,F1 Score (Adv),FP (Adv),ASR
0,DNN,175412,99.9%,322,FGSM,1,100.0%,69,0.1%
1,DNN,175412,99.9%,322,FGSM,5,100.0%,69,0.1%
2,DT,175412,100.0%,99,FGSM,1,99.9%,137,0.1%
3,DT,175412,100.0%,99,FGSM,5,99.9%,137,0.1%
4,RF,175412,100.0%,97,FGSM,1,100.0%,48,0.0%
5,RF,175412,100.0%,97,FGSM,5,100.0%,48,0.0%
6,ET,175412,100.0%,97,FGSM,1,100.0%,47,0.0%
7,ET,175412,100.0%,97,FGSM,5,100.0%,47,0.0%
8,XGBoost,175412,100.0%,115,FGSM,1,99.9%,137,0.1%
9,XGBoost,175412,100.0%,115,FGSM,5,99.9%,137,0.1%


In [ ]:
for key, df in all_results_fp.items():
    df = df.rename(columns={"ASR": "ASR (FP)"})
    df.to_csv(f"results/{key}_fp.csv", index=False)
